# Mangrove Pulse: supported contact-sensing envelope

This notebook checks conditional equations while retaining unknown design inputs. Run all cells to download results or a diagnostics ZIP. The [original executed notebook](Mangrove_Pulse_Supported_Sensing_Envelope_Executed.ipynb) preserves the recorded run; this source copy has revised explanatory prose and unchanged code.

The implemented historical Khor Kalba EO review supplies traceable questions, not present-day sampling stations. A separately dated water-column conductivity/temperature question motivates considering contact sensing. An amphibious platform is one proposed method alongside shore, boat, logger, USV and ordinary hovering probe delivery. No method is selected or authorized by this notebook.

Necessary static constraints do not establish physical feasibility. Synthetic tests check software and algebra; their artificial values are not the team's drone specifications. Measured aircraft performance, independent 2018 accuracy, current ecological diagnosis and ecological benefit remain unproven.

## 1. Exact inputs and synthetic checks

The module, tests, empty water-question contract and dated preparation rationale are embedded byte-for-byte, with SHA-256 checks. The calculation fetches nothing from the network. The contract contains no station or observation records. Known malformed values are rejected even when other values are missing.

The functions use SI units (angles in degrees) and require supplied positive/finite inputs where applicable. Extreme floating-point overflow or loss to zero is rejected, not interpreted as feasibility or a tie. Boundary equalities are not safety margins. The examples below only illustrate conditional implications.

In [ ]:
MODULE_SOURCE = "\"\"\"Conditional static/resource algebra, never a measured aircraft assessment.\n\nSI units except angles in degrees and acceptance probabilities dimensionless.\nUnknown inputs stay unknown. No acquisition, fitting, simulation or deployment.\nSynthetic verification belongs in the separate Colab companion.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\nfrom numbers import Real\n\n\ndef _number(name, value, lower=None, upper=None, strict_lower=False):\n    if isinstance(value, bool) or not isinstance(value, Real):\n        raise ValueError(f\"{name}: a finite real number is required\")\n    try:\n        value = float(value)\n    except (OverflowError, ValueError) as exc:\n        raise ValueError(f\"{name}: representable finite real number required\") from exc\n    if not math.isfinite(value):\n        raise ValueError(f\"{name}: a finite real number is required\")\n    if lower is not None and (value < lower or (strict_lower and value == lower)):\n        raise ValueError(f\"{name}: invalid lower bound\")\n    if upper is not None and value > upper:\n        raise ValueError(f\"{name}: invalid upper bound\")\n    return value\n\n\ndef _product(name, *factors):\n    \"\"\"Reject overflow and loss of a nonzero product to floating-point zero.\"\"\"\n    result = 1.0\n    for factor in factors:\n        previous = result\n        result *= factor\n        _number(name, result)\n        if previous != 0 and factor != 0 and result == 0:\n            raise ValueError(f\"{name}: numerical underflow\")\n    return result\n\n\ndef _ratio(name, numerator, denominator):\n    result = _number(name, numerator / denominator)\n    if numerator != 0 and result == 0:\n        raise ValueError(f\"{name}: numerical underflow\")\n    return result\n\n\ndef _inputs(spec):\n    \"\"\"Validate supplied values even when a different input is missing.\"\"\"\n    missing, values = [], {}\n    for name, (value, lower, upper, strict) in spec.items():\n        if value is None:\n            missing.append(name)\n        else:\n            values[name] = _number(name, value, lower, upper, strict)\n    return values, missing\n\n\ndef _unknown(missing):\n    return {\"status\": \"unknown\", \"missing_inputs\": missing,\n            \"deployment_authorized\": False, \"empirical_claims_proven\": False}\n\n\ndef static_envelope(m0_kg=None, support_kg=None, displacement_m3=None,\n                    water_density_kg_m3=None, usable_thrust_n=None,\n                    tilt_deg=None, lift_margin=None, gravity_m_s2=None):\n    \"\"\"Necessary calm-water buoyancy and steady vertical-force intersection.\n\n    The usable thrust is sustained/condition-specific, not catalogue maximum.\n    Declare any reserve already applied; do not double-count it in lift_margin.\n    Mass/displacement combinations must correspond to a realizable geometry.\n    \"\"\"\n    values, missing = _inputs({\n        \"m0_kg\": (m0_kg, 0, None, True),\n        \"support_kg\": (support_kg, 0, None, False),\n        \"displacement_m3\": (displacement_m3, 0, None, True),\n        \"water_density_kg_m3\": (water_density_kg_m3, 0, None, True),\n        \"usable_thrust_n\": (usable_thrust_n, 0, None, True),\n        \"tilt_deg\": (tilt_deg, 0, 90, False),\n        \"lift_margin\": (lift_margin, 1, None, False),\n        \"gravity_m_s2\": (gravity_m_s2, 0, None, True),\n    })\n    if tilt_deg is not None and values[\"tilt_deg\"] >= 90:\n        raise ValueError(\"tilt_deg must be less than 90\")\n    if missing:\n        return _unknown(missing)\n    buoyancy_capacity = _product(\"computed_buoyancy_capacity_kg\",\n        values[\"water_density_kg_m3\"], values[\"displacement_m3\"])\n    lift_denominator = _product(\"computed_lift_denominator\",\n        values[\"lift_margin\"], values[\"gravity_m_s2\"])\n    vertical_thrust = _product(\"computed_vertical_thrust_n\",\n        values[\"usable_thrust_n\"], math.cos(math.radians(values[\"tilt_deg\"])))\n    lift_capacity = _ratio(\"computed_lift_capacity_kg\", vertical_thrust, lift_denominator)\n    capacity = min(buoyancy_capacity, lift_capacity)\n    total = _number(\"computed_total_mass_kg\", values[\"m0_kg\"] + values[\"support_kg\"], 0, None, True)\n    return {\"status\": \"conditional_static_constraints_only\",\n            \"buoyancy_capacity_kg\": buoyancy_capacity,\n            \"vertical_lift_capacity_kg\": lift_capacity,\n            \"maximum_support_mass_kg\": capacity - values[\"m0_kg\"],\n            \"specified_total_mass_kg\": total,\n            \"mass_slack_kg\": capacity - total,\n            \"necessary_constraints_satisfied\": total <= capacity,\n            \"stability_proven\": False, \"wet_takeoff_proven\": False,\n            \"deployment_authorized\": False, \"empirical_claims_proven\": False}\n\n\ndef restoring_coefficient(mass_kg=None, gravity_m_s2=None, metacentric_height_m=None):\n    \"\"\"For small heel, restoring torque is -k*phi under hydrostatic assumptions.\n\n    Positive k means a restoring sign, not demonstrated dynamic stability.\n    Requires an applicable body/geometry, fixed displaced volume and calm water.\n    \"\"\"\n    values, missing = _inputs({\n        \"mass_kg\": (mass_kg, 0, None, True),\n        \"gravity_m_s2\": (gravity_m_s2, 0, None, True),\n        \"metacentric_height_m\": (metacentric_height_m, None, None, False),\n    })\n    if missing:\n        return _unknown(missing)\n    height = values[\"metacentric_height_m\"]\n    sign = \"restoring\" if height > 0 else \"destabilizing\" if height < 0 else \"neutral_linear_order\"\n    return {\"status\": \"conditional_small_angle_hydrostatic_sign_only\",\n            \"coefficient_n_m_per_rad\": _product(\"computed_restoring_coefficient\",\n                values[\"mass_kg\"], values[\"gravity_m_s2\"], height),\n            \"torque_linearization\": \"torque = -coefficient * heel_radians\",\n            \"sign\": sign, \"dynamic_stability_proven\": False,\n            \"deployment_authorized\": False, \"empirical_claims_proven\": False}\n\n\ndef dwell_advantage_interval(hover_fixed_j=None, supported_fixed_j=None,\n                             hover_dwell_w=None, supported_dwell_w=None,\n                             attempted_tasks=None):\n    \"\"\"Strict supported energy-advantage interval for equal dwell per task.\n\n    EH = hover_fixed_j + n*hover_dwell_w*t, and similarly ES. Fixed energy\n    includes transit, transitions, overhead and recovery under the stated model.\n    Assumes constant powers/fixed costs and equal accepted observations.\n    Power/quality/drift/transition effects that vary with dwell need another model.\n    \"\"\"\n    values, missing = _inputs({\n        \"hover_fixed_j\": (hover_fixed_j, 0, None, False),\n        \"supported_fixed_j\": (supported_fixed_j, 0, None, False),\n        \"hover_dwell_w\": (hover_dwell_w, 0, None, False),\n        \"supported_dwell_w\": (supported_dwell_w, 0, None, False),\n        \"attempted_tasks\": (attempted_tasks, 1, 9007199254740991, False),\n    })\n    if attempted_tasks is not None and not values[\"attempted_tasks\"].is_integer():\n        raise ValueError(\"attempted_tasks must be an integer\")\n    if missing:\n        return _unknown(missing)\n    intercept = values[\"supported_fixed_j\"] - values[\"hover_fixed_j\"]\n    slope = _product(\"computed_energy_slope_j_per_s\",\n        values[\"attempted_tasks\"], values[\"supported_dwell_w\"] - values[\"hover_dwell_w\"])\n    interval, boundary = None, None\n    if slope == 0:\n        if intercept < 0:\n            interval = {\"lower_s\": 0.0, \"lower_inclusive\": True, \"upper_s\": None}\n    else:\n        boundary = _ratio(\"computed_equality_dwell_s\", -intercept, slope)\n        if slope < 0:\n            interval = {\"lower_s\": max(0.0, boundary),\n                        \"lower_inclusive\": boundary < 0, \"upper_s\": None}\n        elif boundary > 0:\n            interval = {\"lower_s\": 0.0, \"lower_inclusive\": True,\n                        \"upper_s\": boundary, \"upper_inclusive\": False}\n    return {\"status\": \"conditional_affine_energy_comparison_only\",\n            \"delta_energy_intercept_j\": intercept,\n            \"delta_energy_slope_j_per_s\": slope,\n            \"algebraic_equality_dwell_s\": boundary,\n            \"strict_supported_advantage_interval\": interval,\n            \"equal_every_dwell\": slope == 0 and intercept == 0,\n            \"acceptance_quality_equal_assumed\": True,\n            \"deployment_authorized\": False, \"empirical_claims_proven\": False}\n\n\ndef accepted_resource_comparison(hover_energy_j=None, supported_energy_j=None,\n                                hover_time_s=None, supported_time_s=None,\n                                hover_acceptance=None, supported_acceptance=None):\n    \"\"\"Compare ratios of expected whole-mission resources to accepted counts.\n\n    Probability lists describe matched attempted tasks. Linearity of expectation\n    gives sum(p_i); independence is not required. Lists are assumptions unless\n    separately supported by measurements. Zero expected successes is undefined.\n    This is not E[energy / accepted_count] for a single mission.\n    \"\"\"\n    values, missing = _inputs({\n        \"hover_energy_j\": (hover_energy_j, 0, None, True),\n        \"supported_energy_j\": (supported_energy_j, 0, None, False),\n        \"hover_time_s\": (hover_time_s, 0, None, True),\n        \"supported_time_s\": (supported_time_s, 0, None, False),\n    })\n    probabilities = {}\n    for name, supplied in ((\"hover_acceptance\", hover_acceptance),\n                           (\"supported_acceptance\", supported_acceptance)):\n        if supplied is None:\n            missing.append(name)\n        elif not isinstance(supplied, (list, tuple)) or not supplied:\n            raise ValueError(f\"{name}: nonempty probability list required\")\n        elif any(value is None for value in supplied):\n            for i, value in enumerate(supplied):\n                if value is not None:\n                    _number(f\"{name}[{i}]\", value, 0, 1)\n            missing.append(f\"{name}: one or more task probabilities unknown\")\n        else:\n            probabilities[name] = [_number(f\"{name}[{i}]\", value, 0, 1)\n                                   for i, value in enumerate(supplied)]\n    if hover_acceptance is not None and supported_acceptance is not None:\n        if len(hover_acceptance) != len(supported_acceptance):\n            raise ValueError(\"Acceptance lists must represent the same matched task count\")\n    if missing:\n        return _unknown(missing)\n    count_h = math.fsum(probabilities[\"hover_acceptance\"])\n    count_s = math.fsum(probabilities[\"supported_acceptance\"])\n    result = {\"status\": \"conditional_ratio_of_expectations_only\",\n              \"hover_expected_accepted_count\": count_h,\n              \"supported_expected_accepted_count\": count_s,\n              \"deployment_authorized\": False, \"empirical_claims_proven\": False}\n    if count_h == 0 or count_s == 0:\n        result.update(status=\"undefined_zero_expected_accepted_count\",\n                      energy_advantage=None, time_advantage=None, both_advantages=None)\n        return result\n    energy_ratio = _ratio(\"computed_energy_ratio\", values[\"supported_energy_j\"], values[\"hover_energy_j\"])\n    time_ratio = _ratio(\"computed_time_ratio\", values[\"supported_time_s\"], values[\"hover_time_s\"])\n    acceptance_ratio = _ratio(\"computed_accepted_count_ratio\", count_s, count_h)\n    result.update(\n        expected_accepted_count_ratio=acceptance_ratio,\n        energy_ratio=energy_ratio, time_ratio=time_ratio,\n        strict_both_advantage_required_count_ratio=max(energy_ratio, time_ratio),\n        hover_energy_j_per_expected_accepted=_ratio(\"computed_hover_energy_per_accepted\", values[\"hover_energy_j\"], count_h),\n        supported_energy_j_per_expected_accepted=_ratio(\"computed_supported_energy_per_accepted\", values[\"supported_energy_j\"], count_s),\n        hover_time_s_per_expected_accepted=_ratio(\"computed_hover_time_per_accepted\", values[\"hover_time_s\"], count_h),\n        supported_time_s_per_expected_accepted=_ratio(\"computed_supported_time_per_accepted\", values[\"supported_time_s\"], count_s),\n        energy_advantage=acceptance_ratio > energy_ratio,\n        time_advantage=acceptance_ratio > time_ratio,\n        both_advantages=acceptance_ratio > max(energy_ratio, time_ratio),\n        strict_both_advantage_possible_at_hover_count=max(energy_ratio, time_ratio)\n            < len(probabilities[\"supported_acceptance\"]) / count_h,\n    )\n    return result\n\n\ndef unknown_design_assessment():\n    \"\"\"Default output deliberately contains no numerical design assumption.\"\"\"\n    return {\"static_envelope\": static_envelope(),\n            \"restoring_sign\": restoring_coefficient(),\n            \"dwell_advantage\": dwell_advantage_interval(),\n            \"accepted_resources\": accepted_resource_comparison(),\n            \"claims\": {\"aircraft_feasibility\": \"unproven\",\n                       \"drone_superiority\": \"unproven\",\n                       \"independent_2018_accuracy\": \"unproven\",\n                       \"ecological_benefit\": \"unproven\"},\n            \"deployment_authorized\": False}\n"
TEST_SOURCE = "\"\"\"Synthetic equation/domain verification only; not aircraft or field tests.\"\"\"\nimport math\nimport unittest\n\nfrom supported_sensing_envelope import (\n    accepted_resource_comparison, dwell_advantage_interval,\n    restoring_coefficient, static_envelope, unknown_design_assessment,\n)\n\n\nclass SupportedEnvelopeTests(unittest.TestCase):\n    def test_unknown_defaults_remain_unknown(self):\n        result = unknown_design_assessment()\n        for key in (\"static_envelope\", \"restoring_sign\", \"dwell_advantage\", \"accepted_resources\"):\n            self.assertEqual(result[key][\"status\"], \"unknown\")\n            self.assertTrue(result[key][\"missing_inputs\"])\n        self.assertFalse(result[\"deployment_authorized\"])\n        self.assertTrue(all(value == \"unproven\" for value in result[\"claims\"].values()))\n\n    def test_buoyancy_limited_envelope(self):\n        result = static_envelope(2, 1, .004, 1000, 100, 0, 1, 10)\n        self.assertEqual(result[\"maximum_support_mass_kg\"], 2)\n        self.assertEqual(result[\"mass_slack_kg\"], 1)\n        self.assertTrue(result[\"necessary_constraints_satisfied\"])\n        self.assertFalse(result[\"stability_proven\"])\n\n    def test_lift_limited_envelope(self):\n        result = static_envelope(2, 1, .020, 1000, 40, 0, 2, 10)\n        self.assertEqual(result[\"maximum_support_mass_kg\"], 0)\n        self.assertFalse(result[\"necessary_constraints_satisfied\"])\n\n    def test_boundary_is_only_necessary_equality(self):\n        result = static_envelope(2, 2, .004, 1000, 100, 0, 1, 10)\n        self.assertEqual(result[\"mass_slack_kg\"], 0)\n        self.assertTrue(result[\"necessary_constraints_satisfied\"])\n        self.assertFalse(result[\"wet_takeoff_proven\"])\n        self.assertFalse(result[\"deployment_authorized\"])\n\n    def test_tilt_and_margin_reduce_lift_capacity(self):\n        result = static_envelope(1, 0, .020, 1000, 100, 60, 2, 10)\n        self.assertAlmostEqual(result[\"vertical_lift_capacity_kg\"], 2.5)\n\n    def test_empty_envelope_rejects_specified_design(self):\n        result = static_envelope(5, 0, .004, 1000, 100, 0, 1, 10)\n        self.assertEqual(result[\"maximum_support_mass_kg\"], -1)\n        self.assertFalse(result[\"necessary_constraints_satisfied\"])\n\n    def test_invalid_domains_and_nonfinite(self):\n        for bad in (True, \"1\", math.inf, math.nan):\n            with self.subTest(bad=bad), self.assertRaises(ValueError):\n                static_envelope(m0_kg=bad)\n        for key, bad in ((\"support_kg\", -1), (\"lift_margin\", .9),\n                         (\"gravity_m_s2\", 0), (\"tilt_deg\", 90), (\"tilt_deg\", -1)):\n            with self.subTest(key=key), self.assertRaises(ValueError):\n                static_envelope(**{key: bad})\n\n    def test_missing_input_does_not_hide_invalid_input(self):\n        with self.assertRaises(ValueError):\n            dwell_advantage_interval(hover_fixed_j=None, supported_dwell_w=-1)\n\n    def test_numerical_overflow_cannot_be_reported_as_a_design_result(self):\n        with self.assertRaises(ValueError):\n            static_envelope(1, 0, 1e308, 1e308, 100, 0, 1, 10)\n        with self.assertRaises(ValueError):\n            restoring_coefficient(1e308, 1e308, .1)\n        with self.assertRaises(ValueError):\n            dwell_advantage_interval(100, 200, 1e308, 0, 100)\n        with self.assertRaises(ValueError):\n            accepted_resource_comparison(1e308, 50, 100, 50, [1e-308], [1])\n        with self.assertRaises(ValueError):\n            static_envelope(m0_kg=10 ** 1000)\n\n    def test_numerical_underflow_does_not_create_a_false_tie(self):\n        with self.assertRaises(ValueError):\n            restoring_coefficient(1e-200, 1e-200, .1)\n        with self.assertRaises(ValueError):\n            accepted_resource_comparison(1e308, 1e-308, 100, 50, [1], [1])\n\n    def test_restoring_sign_and_dimensions(self):\n        result = restoring_coefficient(2, 10, .1)\n        self.assertEqual(result[\"coefficient_n_m_per_rad\"], 2)\n        self.assertEqual(result[\"sign\"], \"restoring\")\n        self.assertFalse(result[\"dynamic_stability_proven\"])\n\n    def test_neutral_and_destabilizing_are_not_passes(self):\n        self.assertEqual(restoring_coefficient(2, 10, 0)[\"sign\"], \"neutral_linear_order\")\n        self.assertEqual(restoring_coefficient(2, 10, -.1)[\"sign\"], \"destabilizing\")\n\n    def test_positive_break_even_dwell(self):\n        result = dwell_advantage_interval(100, 400, 10, 2, 3)\n        self.assertEqual(result[\"algebraic_equality_dwell_s\"], 12.5)\n        self.assertEqual(result[\"strict_supported_advantage_interval\"],\n                         {\"lower_s\": 12.5, \"lower_inclusive\": False, \"upper_s\": None})\n\n    def test_short_dwell_can_win_when_supported_power_is_higher(self):\n        result = dwell_advantage_interval(400, 100, 2, 10, 3)\n        self.assertEqual(result[\"strict_supported_advantage_interval\"],\n                         {\"lower_s\": 0.0, \"lower_inclusive\": True,\n                          \"upper_s\": 12.5, \"upper_inclusive\": False})\n\n    def test_all_nonnegative_dwell_advantage(self):\n        result = dwell_advantage_interval(400, 100, 10, 2, 3)\n        self.assertEqual(result[\"strict_supported_advantage_interval\"],\n                         {\"lower_s\": 0.0, \"lower_inclusive\": True, \"upper_s\": None})\n\n    def test_same_power_offset_and_tie(self):\n        self.assertTrue(dwell_advantage_interval(100, 100, 2, 2, 1)[\"equal_every_dwell\"])\n        self.assertIsNone(dwell_advantage_interval(100, 200, 2, 2, 1)[\"strict_supported_advantage_interval\"])\n        self.assertIsNotNone(dwell_advantage_interval(200, 100, 2, 2, 1)[\"strict_supported_advantage_interval\"])\n\n    def test_zero_boundary_is_not_strict_advantage_at_zero(self):\n        result = dwell_advantage_interval(100, 100, 10, 2, 1)\n        self.assertFalse(result[\"strict_supported_advantage_interval\"][\"lower_inclusive\"])\n        self.assertIsNone(dwell_advantage_interval(100, 100, 2, 10, 1)[\"strict_supported_advantage_interval\"])\n\n    def test_attempt_count_is_positive_integer(self):\n        for bad in (0, -1, 1.5, True):\n            with self.subTest(bad=bad), self.assertRaises(ValueError):\n                dwell_advantage_interval(attempted_tasks=bad)\n\n    def test_interval_matches_direct_energy_on_synthetic_grid(self):\n        for eh0, es0, ph, ps, n in ((100, 400, 10, 2, 3), (400, 100, 2, 10, 3),\n                                   (100, 100, 2, 2, 1), (200, 100, 2, 2, 2),\n                                   (100, 100, 10, 2, 1), (400, 100, 10, 2, 3)):\n            interval = dwell_advantage_interval(eh0, es0, ph, ps, n)[\"strict_supported_advantage_interval\"]\n            for dwell in (0, 1, 12.5, 13, 100):\n                expected = es0 + n * ps * dwell < eh0 + n * ph * dwell\n                inside = False\n                if interval:\n                    inside = dwell > interval[\"lower_s\"] or (\n                        interval[\"lower_inclusive\"] and dwell == interval[\"lower_s\"])\n                    if interval[\"upper_s\"] is not None:\n                        inside = inside and dwell < interval[\"upper_s\"]\n                self.assertEqual(inside, expected)\n\n    def test_energy_advantage_need_not_be_time_advantage(self):\n        result = accepted_resource_comparison(100, 75, 100, 112, [.8, .8], [.8, .8])\n        self.assertTrue(result[\"energy_advantage\"])\n        self.assertFalse(result[\"time_advantage\"])\n        self.assertFalse(result[\"both_advantages\"])\n\n    def test_heterogeneous_probabilities_use_expected_counts(self):\n        result = accepted_resource_comparison(100, 50, 100, 50, [.1, .9], [.4, .4])\n        self.assertAlmostEqual(result[\"hover_expected_accepted_count\"], 1)\n        self.assertAlmostEqual(result[\"supported_expected_accepted_count\"], .8)\n        self.assertAlmostEqual(result[\"supported_energy_j_per_expected_accepted\"], 62.5)\n        self.assertTrue(result[\"both_advantages\"])\n\n    def test_probability_ceiling_can_make_both_advantages_impossible(self):\n        result = accepted_resource_comparison(100, 75, 100, 112, [.95, .95], [1, 1])\n        self.assertFalse(result[\"strict_both_advantage_possible_at_hover_count\"])\n        self.assertFalse(result[\"both_advantages\"])\n\n    def test_zero_success_is_undefined_not_zero_resource(self):\n        for ph, ps in (([0], [0]), ([0], [.5]), ([.5], [0])):\n            result = accepted_resource_comparison(100, 50, 100, 50, ph, ps)\n            self.assertEqual(result[\"status\"], \"undefined_zero_expected_accepted_count\")\n            self.assertIsNone(result[\"both_advantages\"])\n\n    def test_invalid_and_mismatched_probability_records(self):\n        for ph, ps in (([], []), ([1], [1, 1]), ([1.1], [1]), ([True], [1]),\n                       ([-.1], [1]), ([math.nan], [1]), (\"bad\", [1])):\n            with self.subTest(ph=ph), self.assertRaises(ValueError):\n                accepted_resource_comparison(100, 50, 100, 50, ph, ps)\n\n    def test_unknown_probability_cannot_be_filled_with_success(self):\n        result = accepted_resource_comparison(100, 50, 100, 50, [.5, None], [.5, .5])\n        self.assertEqual(result[\"status\"], \"unknown\")\n        self.assertNotIn(\"both_advantages\", result)\n\n    def test_resource_comparison_matches_direct_ratios(self):\n        for es, ts, ph, ps in ((75, 112, .8, .8), (150, 75, .5, .8), (50, 50, .9, .2)):\n            result = accepted_resource_comparison(100, es, 100, ts, [ph], [ps])\n            self.assertEqual(result[\"energy_advantage\"], es / ps < 100 / ph)\n            self.assertEqual(result[\"time_advantage\"], ts / ps < 100 / ph)\n            self.assertFalse(result[\"empirical_claims_proven\"])\n\n\nif __name__ == \"__main__\":\n    unittest.main()\n"
QUESTION_TEXT = "{\n  \"schema_version\": \"1.0\",\n  \"question_id\": \"MP-KALBA-WATER-CONTEXT-PROPOSED-01\",\n  \"status\": \"proposed_no_observations\",\n  \"question\": \"How do water-column conductivity and temperature vary between authorized open-channel and accessible interior-water locations, with depth and tidal context recorded?\",\n  \"site\": \"Khor Kalba, UAE\",\n  \"aoi_is_official_reserve_boundary\": false,\n  \"motivation\": {\n    \"source\": \"https://rsis.ramsar.org/RISapp/files/RISrep/AE2125RIS.pdf\",\n    \"document_completed\": \"2012-09-01\",\n    \"page\": 5,\n    \"source_type\": \"historical_official_site_description\",\n    \"measurement_dates\": null,\n    \"sample_count\": null,\n    \"does_not_establish\": [\"present_harm\", \"salinity_harm_threshold\", \"planting_need\", \"aircraft_requirement\"]\n  },\n  \"relationship_to_frozen_2018_cases\": \"Site-level motivation for a separate dated experiment, not validation of the historical cells or a conversion of audit centres into sampling stations.\",\n  \"observation_date\": null,\n  \"stations\": [],\n  \"stations_are_flight_or_landing_waypoints\": false,\n  \"authorization\": null,\n  \"operator_and_recovery_protocol\": null,\n  \"sensor\": {\n    \"model\": null,\n    \"calibration\": null,\n    \"conductivity_units\": null,\n    \"temperature_units\": \"degree_Celsius\",\n    \"salinity_conversion\": null,\n    \"salinity_units\": null,\n    \"response_and_stability_rule\": null,\n    \"accuracy_requirement_and_provenance\": null\n  },\n  \"required_context\": [\"time_utc\", \"position_and_uncertainty\", \"sampling_depth_and_uncertainty\", \"tide_or_water_level_context\", \"sensor_identity\", \"calibration_record\", \"measurement_stability\", \"platform_disturbance\", \"rejection_reason\"],\n  \"comparators_to_assess\": [\"shore_probe\", \"boat_probe\", \"fixed_logger\", \"uncrewed_surface_vehicle\", \"ordinary_hovering_probe_delivery\", \"water_supported_probe_delivery\"],\n  \"common_acceptance_rule\": null,\n  \"matched_trial_design\": null,\n  \"whole_mission_resources\": [\"energy\", \"elapsed_time\", \"operator_effort\", \"capital_and_operating_cost\", \"failed_attempts\", \"recovery\", \"accepted_observation_count\"],\n  \"observations\": [],\n  \"accepted_observation_count\": 0,\n  \"not_inferred\": [\"root_zone_porewater\", \"canopy_stress_cause\", \"carbon\", \"restoration_suitability\", \"planting_need\", \"historical_cover_truth\", \"drone_superiority\"],\n  \"deployment_authorized\": false\n}\n"
RATIONALE_TEXT = "# A locally grounded EO-to-contact-sensing question\n\n7 October 2026 | Evidence-backed research rationale | No water observations or aircraft tests collected\n\n## The concrete connection\n\n**Question:** how do water-column conductivity and temperature vary between authorized open-channel and accessible interior-water locations, with depth and tidal context recorded?\n\nThis is a proposed, separately dated task. [Khor Kalba's official Ramsar record](https://rsis.ramsar.org/RISapp/files/RISrep/AE2125RIS.pdf), completed 1 September 2012, page 5, reports historical salinity differences between beach water (34-38 ppt) and the mangrove interior (38-44 ppt), with temperature variation. The section gives no measurement dates or sample count. It motivates investigation, not a current harm diagnosis or a salinity threshold for planting.\n\n| Link | What it supplies | What it does not supply |\n|---|---|---|\n| Frozen 2018 EO/reference review | Demonstrated evidence/date/quality handling and traceable unresolved cover questions | Current cover, current water properties or independent historical truth |\n| Separate dated EO context | Proposed visible water/vegetation context and explicit optical ambiguity | Water-column conductivity, sensor depth, navigable water or landing permission |\n| Contact observation | Proposed calibrated conductivity/temperature, position, time, depth and tide/water-level records | Root-zone porewater, stress cause, carbon or planting suitability |\n| Platform comparison | Proposed quality-matched whole-mission energy/time/cost and recovery comparison | A predetermined amphibious winner |\n| Qualified evidence review | A future interpretation of accepted measurements against a defined question | Automatic intervention or established ecological benefit |\n\nThe local motivation applies to the site, not automatically to C027's historical excluded centres. Those 42 centres are not present sampling stations. C001/C017 photographic questions still have photographic comparators; contact sensing answers a different observable. No contemporary station or route is specified.\n\n## Which data and tools earn a place?\n\n- **Use:** the already demonstrated Sentinel-2/GMW baseline and saved WorldCover context, preserving dates and shared-reference limitations.\n- **Next EO input:** Planetary Computer returns a covering candidate Sentinel-2C item `S2C_MSIL2A_20260924T064621_R020_T40RDN_20260924T114716` (24 September 2026; PB 05.13). No new pixels have been processed. Per-band encoding must be checked before computing calibrated indices; scene cloud percentage is not local quality.\n- **Hyperspectral candidate:** DLR's [EnMAP catalogue](https://geoservice.dlr.de/eoc/ogc/stac/) returns item `ENMAP01-____L2A-DT0000152124_20250911T072327Z_006_V010502_20250914T181224Z`, acquired 11 September 2025: 224 advertised bands, 30 m grid, scene cloud 19%. This establishes a catalogue intersection, not valid local pixels or a useful hyperspectral result. Data-asset requests returned login HTML, not the image/product XML. Authenticate lawfully and check correction/QA/rights before an experiment. The [public ARD default](https://www.enmap.org/data_access/) is land-mode correction, not an automatically valid aquatic product.\n- **Do not substitute unrelated training data:** none of the 43 individual bboxes in the checked [Planet coastal-water collection](https://www.planet.com/data/stac/tanager-core-imagery/coastal-water-bodies/collection.json) intersects this AOI. Its union bbox and the starter notebook's UAE wording do not establish Kalba coverage. Other Planet collections were not ruled out.\n- **Keep optional:** Landsat for a defined longer-term contextual question; GBIF for appropriate biodiversity context, not absence/species/health truth at these cells; geojson.io for geometry inspection, not navigation validation. Add them only where they change the evidence or decision. No 813 covering scene/date was provided by the inspected Cockpit card.\n\nAny new raster acquisition/processing belongs in a separate Colab experiment. Preserve the original threshold, split and results. Do not mix the 2025 hyperspectral scene, recent EO and 2018 cover into a single accuracy or change claim. Redistribution of EnMAP source pixels requires a check of the [applicable licence](https://geoservice.dlr.de/resources/licenses/enmap/EnMAP-Data_License_v1_1.pdf); they are not bundled here.\n\n**Anonymous-access check, 8 October:** a bounded image-URL request returned HTTP 200 with `text/html;charset=UTF-8` and an HTML prefix, not raster bytes. The official [no-registration examples](https://www.enmap.org/data_tools/exampledata/) cover Tyrol, Groningen and Nevada, not this site. The [EnMAP-Box access documentation](https://enmap-box.readthedocs.io/en/latest/general/data_access.html) describes a registration exemption request through `enmap_registration@dlr.de` for people without institutional email. No request has been sent, exemption approved or account created. Public metadata and unrelated samples cannot substitute for local spectral observations.\n\n## Why amphibious support is a testable improvement, not a decorative feature\n\nThe candidate mechanism is water-supported dwell: transport a suitable contact probe by air, obtain stationary accepted observations while supported, then recover it. This might reduce airborne dwell, but support mass, drift, transitions, sensing bias and failed recovery can offset the benefit. The team has not implemented this mechanism.\n\nAn [ordinary drone/boat comparison published in 2026](https://doi.org/10.1007/s10661-026-15473-0) provides an important alternative: three Irish lakes, 27 paired sonde comparisons, estimated sampling times of 28 versus 45 minutes, but higher initial drone cost and variable-specific discrepancies. It is freshwater, not an amphibious or Khor Kalba trial. Nonsignificant differences do not prove equivalence. Our comparison must include ordinary hovering probe delivery, not assume all ordinary drones only take photographs.\n\nChoose among shore/boat probes, fixed loggers, USVs, ordinary hover delivery and supported delivery for the **same observation contract**. Prefer the least burdensome method that meets its acceptance and habitat/access constraints. No permissions, calibrated payload or safe water operation are established by this document.\n\n## Conditional analytical claims that withstand scrutiny\n\nWith aircraft/payload mass `m0`, support mass `mf`, allowable displacement `Vallow`, water density `rho_w`, condition-specific usable thrust `Tusable`, tilt `theta`, declared lift margin `gamma >= 1`, and gravity `g`, simultaneous static buoyancy and vertical-lift constraints imply:\n\n`mf <= min(rho_w * Vallow, Tusable * cos(theta) / (gamma * g)) - m0`.\n\nThis is a necessary design envelope, not a selected craft's performance. Inputs are unknown here, support geometry/mass are coupled, and any reserve already included in `Tusable` must not be counted twice. [Buoyancy](https://openstax.org/books/university-physics-volume-1/pages/14-4-archimedes-principle-and-buoyancy) does not establish [small-angle restoring stability](https://farside.ph.utexas.edu/teaching/336L/Fluidhtml/node29.html), wet takeoff or recovery. An empty envelope rejects that specified design; a nonempty envelope does not validate it.\n\nFor equal attempted tasks, common per-attempt acceptance rates `pH,pS > 0`, and expected whole-mission resources `EH,ES,TH,TS`, supported sensing improves both aggregate energy and time per accepted record only if:\n\n`pS/pH > max(ES/EH, TS/TH)`.\n\nThis follows directly from comparing ratios of expectations; it is not an expected per-mission ratio. For heterogeneous tasks use the expected accepted count `sum(p_i)`. The earlier illustrative main case saves energy but takes longer at equal assumed acceptance. No measured acceptance rates, resource advantage or superiority over a UAE product are supplied.\n\nThe separate [supported-sensing envelope notebook](../supplementary/Mangrove_Pulse_Supported_Sensing_Envelope.ipynb) implements these conditional screens, strict dwell-advantage boundaries and explicit counterexamples using standard-library Python only. It embeds the empty question contract, source and 26 synthetic test definitions; it needs no additional dataset upload. It is currently an unexecuted source companion. Static source checks and a bounded read-only review are not a successful hosted run or physical validation. Its default aircraft inputs remain unknown.\n\n## Acceptance before interpretation\n\nThe [blank water-context question contract](../examples/inspection/water_context_question.json) requires a separate date, authorized station definition, selected calibrated sensor, units/conversion, depth/time/position/tidal context, rejection rules and matched comparator. Its station list and observations are empty. No arbitrary success tolerance is populated without a sensor specification and decision requirement.\n\nWater-column salinity is not porewater salinity. A colour/index is not chlorophyll concentration, harmful bloom toxicity or ecological damage. A measurement is useful only if it meets the contract and helps a qualified interpretation. Any planting or ecological-benefit claim still needs the [separate evidence protocol](VALIDATION_AND_BENEFIT_TESTS.md).\n\n**Current result:** a sourced local research question, verified candidate catalogue records and conditional inequalities. **Not yet a result:** local usable hyperspectral pixels, accepted water observations, aircraft feasibility, independent 2018 accuracy or ecological improvement.\n"
EMBEDDED_SHA256 = {"module":"42a254892692e4ab34b2a0138555f45ce1af21affe320f712868259eb73977f7","tests":"1c1945def9cae6d820bedec3c8891b2eef8e5d04b99f5abb8009370f3d9dd7fd","question":"f60e0d8e2e7e8634ecc5400acc0222adc33a1efae00c26cc342bd7c65e5dbefb","rationale":"cb0f5d763433e462590f123fca0c5330ce9b657f5b1a2318b7df464c13bdeca7"}

import contextlib
import hashlib
import io
import json
import sys
import traceback
import types
import unittest

errors = []
test_log = io.StringIO()
result = None
module = None
prepared = False
embedded = {'module': MODULE_SOURCE, 'tests': TEST_SOURCE,
            'question': QUESTION_TEXT, 'rationale': RATIONALE_TEXT}
try:
    for name, content in embedded.items():
        assert hashlib.sha256(content.encode('utf-8')).hexdigest() == EMBEDDED_SHA256[name], name
    question = json.loads(QUESTION_TEXT)
    assert question['stations'] == [] and question['observations'] == []
    assert question['accepted_observation_count'] == 0
    assert question['deployment_authorized'] is False
    module = types.ModuleType('supported_sensing_envelope')
    exec(MODULE_SOURCE, module.__dict__)
    sys.modules[module.__name__] = module
    test_module = types.ModuleType('test_supported_sensing_envelope')
    exec(TEST_SOURCE, test_module.__dict__)
    suite = unittest.defaultTestLoader.loadTestsFromModule(test_module)
    result = unittest.TextTestRunner(stream=test_log, verbosity=2).run(suite)
    prepared = result.wasSuccessful()
    if not prepared:
        errors.append('Synthetic test failure; no assessment treated as successful.')
except Exception:
    errors.append(traceback.format_exc())
print(test_log.getvalue())
print('Preparation:', 'synthetic checks passed' if prepared else 'failed; run final cell for diagnostics')
print('No measured aircraft, independent cover labels or ecological observations were tested.')


## 2. Conditional claims and counterexamples

For aircraft/payload mass `m0`, support mass `mf`, allowed displacement `V`, water density `rho`, condition-specific usable thrust `T`, tilt `theta`, declared margin `gamma >= 1`, and gravity `g`:

`mf <= min(rho*V, T*cos(theta)/(gamma*g)) - m0`.

This intersects buoyancy and steady vertical force. Geometry, freeboard, reserve displacement, power/thermal capacity, changing payload, hull stability, drag, gusts, waves, wet takeoff and recovery are separate obligations. Support mass and displacement cannot be chosen independently without a realizable design.

Under applicable calm-water, small-angle hydrostatics, `torque = -m*g*GM*phi`. A positive metacentric height gives a restoring sign, not validated forced/damped dynamics or disturbance tolerance. Euler's equations describe dynamics only after inertia, forcing, damping and controls are specified; those measurements are missing for this concept.

For constant-power, equal-dwell, equal-accepted-quality comparison, whole-mission energy difference is `ES - EH = (ES0 - EH0) + n*(PS - PH)*t`. The code solves its strict advantage interval, including no-advantage and boundary cases. Fixed costs must include transit, support transitions and recovery; a low supported dwell power alone does not establish a mission advantage.

For expected accepted counts `NH=sum(pH_i)>0` and `NS=sum(pS_i)>0`, both aggregate energy and time per accepted record improve only if `NS/NH > max(ES/EH, TS/TH)`. Independence of attempts is not needed for expected counts. This compares ratios of expectations, not the expectation of a per-mission ratio. Acceptance probabilities and resources remain assumptions until measured under a common contract.

Sources: [Archimedes' principle](https://openstax.org/books/university-physics-volume-1/pages/14-4-archimedes-principle-and-buoyancy), [small-angle floating-body stability](https://farside.ph.utexas.edu/teaching/336L/Fluidhtml/node29.html), and the independently dated source rationale below. The artificial counterexample deliberately shows energy benefit without time benefit.

In [ ]:
assessment = None
synthetic_examples = None
if prepared:
    try:
        assessment = module.unknown_design_assessment()
        assert assessment['deployment_authorized'] is False
        assert all(value == 'unproven' for value in assessment['claims'].values())
        synthetic_examples = {
            'evidence_kind': 'artificial_equation_demonstrations_not_team_hardware',
            'energy_without_time_advantage': module.accepted_resource_comparison(
                100, 75, 100, 112, [.8, .8], [.8, .8]),
            'supported_extra_fixed_cost_needs_dwell': module.dwell_advantage_interval(
                100, 400, 10, 2, 3),
            'support_mass_violates_lift_envelope': module.static_envelope(
                2, 1, .020, 1000, 40, 0, 2, 10),
        }
        assert synthetic_examples['energy_without_time_advantage']['energy_advantage'] is True
        assert synthetic_examples['energy_without_time_advantage']['both_advantages'] is False
        assert synthetic_examples['support_mass_violates_lift_envelope']['necessary_constraints_satisfied'] is False
        print('Actual concept assessment: no new numerical design assumptions')
        print(json.dumps(assessment, indent=2, allow_nan=False))
        print('SYNTHETIC demonstrations, not physical observations')
        print(json.dumps(synthetic_examples, indent=2, allow_nan=False))
    except Exception:
        errors.append(traceback.format_exc())
        assessment = None
        synthetic_examples = None
else:
    print('Assessment skipped after preparation failure. Run final cell for diagnostics.')


## 3. Archived local-question rationale

The dated preparation rationale below remains unchanged with the embedded inputs. Its acquisition and execution-status statements describe that earlier checkpoint. Read the [current water-contact rationale](../docs/WATER_CONTACT_RATIONALE.md) and [dated EO context](../docs/DATED_EO_CONTEXT.md) for subsequent results and their limits.

# A locally grounded EO-to-contact-sensing question

7 October 2026 | Evidence-backed research rationale | No water observations or aircraft tests collected

## The concrete connection

**Question:** how do water-column conductivity and temperature vary between authorized open-channel and accessible interior-water locations, with depth and tidal context recorded?

This is a proposed, separately dated task. [Khor Kalba's official Ramsar record](https://rsis.ramsar.org/RISapp/files/RISrep/AE2125RIS.pdf), completed 1 September 2012, page 5, reports historical salinity differences between beach water (34-38 ppt) and the mangrove interior (38-44 ppt), with temperature variation. The section gives no measurement dates or sample count. It motivates investigation, not a current harm diagnosis or a salinity threshold for planting.

| Link | What it supplies | What it does not supply |
|---|---|---|
| Frozen 2018 EO/reference review | Demonstrated evidence/date/quality handling and traceable unresolved cover questions | Current cover, current water properties or independent historical truth |
| Separate dated EO context | Proposed visible water/vegetation context and explicit optical ambiguity | Water-column conductivity, sensor depth, navigable water or landing permission |
| Contact observation | Proposed calibrated conductivity/temperature, position, time, depth and tide/water-level records | Root-zone porewater, stress cause, carbon or planting suitability |
| Platform comparison | Proposed quality-matched whole-mission energy/time/cost and recovery comparison | A predetermined amphibious winner |
| Qualified evidence review | A future interpretation of accepted measurements against a defined question | Automatic intervention or established ecological benefit |

The local motivation applies to the site, not automatically to C027's historical excluded centres. Those 42 centres are not present sampling stations. C001/C017 photographic questions still have photographic comparators; contact sensing answers a different observable. No contemporary station or route is specified.

## Which data and tools earn a place?

- **Use:** the already demonstrated Sentinel-2/GMW baseline and saved WorldCover context, preserving dates and shared-reference limitations.
- **Next EO input:** Planetary Computer returns a covering candidate Sentinel-2C item `S2C_MSIL2A_20260924T064621_R020_T40RDN_20260924T114716` (24 September 2026; PB 05.13). No new pixels have been processed. Per-band encoding must be checked before computing calibrated indices; scene cloud percentage is not local quality.
- **Hyperspectral candidate:** DLR's [EnMAP catalogue](https://geoservice.dlr.de/eoc/ogc/stac/) returns item `ENMAP01-____L2A-DT0000152124_20250911T072327Z_006_V010502_20250914T181224Z`, acquired 11 September 2025: 224 advertised bands, 30 m grid, scene cloud 19%. This establishes a catalogue intersection, not valid local pixels or a useful hyperspectral result. Data-asset requests returned login HTML, not the image/product XML. Authenticate lawfully and check correction/QA/rights before an experiment. The [public ARD default](https://www.enmap.org/data_access/) is land-mode correction, not an automatically valid aquatic product.
- **Do not substitute unrelated training data:** none of the 43 individual bboxes in the checked [Planet coastal-water collection](https://www.planet.com/data/stac/tanager-core-imagery/coastal-water-bodies/collection.json) intersects this AOI. Its union bbox and the starter notebook's UAE wording do not establish Kalba coverage. Other Planet collections were not ruled out.
- **Keep optional:** Landsat for a defined longer-term contextual question; GBIF for appropriate biodiversity context, not absence/species/health truth at these cells; geojson.io for geometry inspection, not navigation validation. Add them only where they change the evidence or decision. No 813 covering scene/date was provided by the inspected Cockpit card.

Any new raster acquisition/processing belongs in a separate Colab experiment. Preserve the original threshold, split and results. Do not mix the 2025 hyperspectral scene, recent EO and 2018 cover into a single accuracy or change claim. Redistribution of EnMAP source pixels requires a check of the [applicable licence](https://geoservice.dlr.de/resources/licenses/enmap/EnMAP-Data_License_v1_1.pdf); they are not bundled here.

**Anonymous-access check, 8 October:** a bounded image-URL request returned HTTP 200 with `text/html;charset=UTF-8` and an HTML prefix, not raster bytes. The official [no-registration examples](https://www.enmap.org/data_tools/exampledata/) cover Tyrol, Groningen and Nevada, not this site. The [EnMAP-Box access documentation](https://enmap-box.readthedocs.io/en/latest/general/data_access.html) describes a registration exemption request through `enmap_registration@dlr.de` for people without institutional email. No request has been sent, exemption approved or account created. Public metadata and unrelated samples cannot substitute for local spectral observations.

## Why amphibious support is a testable improvement, not a decorative feature

The candidate mechanism is water-supported dwell: transport a suitable contact probe by air, obtain stationary accepted observations while supported, then recover it. This might reduce airborne dwell, but support mass, drift, transitions, sensing bias and failed recovery can offset the benefit. The team has not implemented this mechanism.

An [ordinary drone/boat comparison published in 2026](https://doi.org/10.1007/s10661-026-15473-0) provides an important alternative: three Irish lakes, 27 paired sonde comparisons, estimated sampling times of 28 versus 45 minutes, but higher initial drone cost and variable-specific discrepancies. It is freshwater, not an amphibious or Khor Kalba trial. Nonsignificant differences do not prove equivalence. Our comparison must include ordinary hovering probe delivery, not assume all ordinary drones only take photographs.

Choose among shore/boat probes, fixed loggers, USVs, ordinary hover delivery and supported delivery for the **same observation contract**. Prefer the least burdensome method that meets its acceptance and habitat/access constraints. No permissions, calibrated payload or safe water operation are established by this document.

## Conditional analytical claims that withstand scrutiny

With aircraft/payload mass `m0`, support mass `mf`, allowable displacement `Vallow`, water density `rho_w`, condition-specific usable thrust `Tusable`, tilt `theta`, declared lift margin `gamma >= 1`, and gravity `g`, simultaneous static buoyancy and vertical-lift constraints imply:

`mf <= min(rho_w * Vallow, Tusable * cos(theta) / (gamma * g)) - m0`.

This is a necessary design envelope, not a selected craft's performance. Inputs are unknown here, support geometry/mass are coupled, and any reserve already included in `Tusable` must not be counted twice. [Buoyancy](https://openstax.org/books/university-physics-volume-1/pages/14-4-archimedes-principle-and-buoyancy) does not establish [small-angle restoring stability](https://farside.ph.utexas.edu/teaching/336L/Fluidhtml/node29.html), wet takeoff or recovery. An empty envelope rejects that specified design; a nonempty envelope does not validate it.

For equal attempted tasks, common per-attempt acceptance rates `pH,pS > 0`, and expected whole-mission resources `EH,ES,TH,TS`, supported sensing improves both aggregate energy and time per accepted record only if:

`pS/pH > max(ES/EH, TS/TH)`.

This follows directly from comparing ratios of expectations; it is not an expected per-mission ratio. For heterogeneous tasks use the expected accepted count `sum(p_i)`. The earlier illustrative main case saves energy but takes longer at equal assumed acceptance. No measured acceptance rates, resource advantage or superiority over a UAE product are supplied.

The separate supported-sensing envelope notebook (repository path: `../supplementary/Mangrove_Pulse_Supported_Sensing_Envelope.ipynb`) implements these conditional screens, strict dwell-advantage boundaries and explicit counterexamples using standard-library Python only. It embeds the empty question contract, source and 26 synthetic test definitions; it needs no additional dataset upload. It is currently an unexecuted source companion. Static source checks and a bounded read-only review are not a successful hosted run or physical validation. Its default aircraft inputs remain unknown.

## Acceptance before interpretation

The blank water-context question contract (repository path: `../examples/inspection/water_context_question.json`) requires a separate date, authorized station definition, selected calibrated sensor, units/conversion, depth/time/position/tidal context, rejection rules and matched comparator. Its station list and observations are empty. No arbitrary success tolerance is populated without a sensor specification and decision requirement.

Water-column salinity is not porewater salinity. A colour/index is not chlorophyll concentration, harmful bloom toxicity or ecological damage. A measurement is useful only if it meets the contract and helps a qualified interpretation. Any planting or ecological-benefit claim still needs the separate evidence protocol (repository path: `VALIDATION_AND_BENEFIT_TESTS.md`).

**Current result:** a sourced local research question, verified candidate catalogue records and conditional inequalities. **Not yet a result:** local usable hyperspectral pixels, accepted water observations, aircraft feasibility, independent 2018 accuracy or ecological improvement.


## 4. Export and interpretation

The final cell writes a unique ZIP containing exact sources, the empty question contract, this preparation rationale, synthetic-test log, unknown assessment, clearly artificial counterexamples, status and payload hashes. A succeeded status means this analytical/software run completed; it does not upgrade any empirical claim. Failures produce a diagnostics ZIP instead. No old results are overwritten.

Next substantive EO work must be a separate dated Colab experiment with calibrated band encoding, local QA, mixed-pixel and aquatic-correction limitations. No local station, landing site, salinity, field label or ecological outcome can be generated from this algebra. EnMAP image access is pending; anonymous catalogue metadata is not spectral data. No public-source pixels or authentication state are bundled.

In [ ]:
from pathlib import Path
import platform
import tempfile
import zipfile

export_root = Path(tempfile.mkdtemp(prefix='mangrove_supported_envelope_'))
status = {
    'experiment_id': 'MP-SUPPORTED-SENSING-ENVELOPE-01',
    'status': 'succeeded' if prepared and assessment is not None and not errors else 'failed_diagnostics_only',
    'evidence_kind': 'conditional_analytic_assessment_and_synthetic_software_tests',
    'python_version': platform.python_version(),
    'synthetic_tests_run': result.testsRun if result is not None else 0,
    'synthetic_tests_passed': result.wasSuccessful() if result is not None else False,
    'actual_drone_missions': 0,
    'independent_2018_labels': 0,
    'water_observations': 0,
    'ecological_outcome_observations': 0,
    'deployment_authorized': False,
    'claims': {name: 'unproven' for name in (
        'aircraft_feasibility', 'drone_superiority', 'independent_2018_accuracy', 'ecological_benefit')},
    'embedded_sha256': EMBEDDED_SHA256,
}
def json_text(value):
    return json.dumps(value, indent=2, allow_nan=False) + '\n'
payloads = {
    'supported_sensing_envelope.py': MODULE_SOURCE,
    'test_supported_sensing_envelope.py': TEST_SOURCE,
    'water_context_question.json': QUESTION_TEXT,
    'water_contact_rationale.md': RATIONALE_TEXT,
    'status.json': json_text(status),
    'unknown_assessment.json': json_text(assessment),
    'synthetic_examples_NOT_MEASUREMENTS.json': json_text(synthetic_examples),
    'synthetic_tests.log': test_log.getvalue(),
    'diagnostics.json': json_text(errors),
    'README.md': '# Conditional analytical companion\n\nRead status.json before using any output. A succeeded status means synthetic software/equation checks and export completed, not aircraft, cover or ecological validation. All empirical claims remain unproven; all actual observation counts are zero. Unknown inputs stay unknown. Synthetic examples are not team designs or missions. No flight, water contact, planting, publication or submission is authorized.\n',
}
for name, content in payloads.items():
    (export_root / name).write_bytes(content.encode('utf-8'))
manifest = {'algorithm': 'sha256', 'files': [
    {'path': name, 'bytes': len(content.encode('utf-8')), 'sha256': hashlib.sha256(content.encode('utf-8')).hexdigest()}
    for name, content in sorted(payloads.items())]}
(export_root / 'manifest.json').write_bytes(json_text(manifest).encode('utf-8'))
zip_path = export_root / 'Mangrove_Pulse_Supported_Sensing_Envelope_Results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in sorted([*payloads, 'manifest.json']):
        archive.write(export_root / name, arcname=name)
with zipfile.ZipFile(zip_path) as archive:
    assert archive.testzip() is None
    for record in manifest['files']:
        data = archive.read(record['path'])
        assert len(data) == record['bytes']
        assert hashlib.sha256(data).hexdigest() == record['sha256']
print(json.dumps(status, indent=2))
print('ZIP:', zip_path.name, 'bytes:', zip_path.stat().st_size,
      'SHA-256:', hashlib.sha256(zip_path.read_bytes()).hexdigest())
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Download fallback:', zip_path)
